# Aethermoor NPC AI — QLoRA Fine-Tuning

Fine-tune Qwen2.5-3B-Instruct on SCBE-AETHERMOORE training data with
dual-manifold personality encoding.

**What this does:**
1. Loads your chat-format training data from HuggingFace
2. QLoRA fine-tunes Qwen2.5-3B on free Colab T4 GPU
3. Pushes the trained adapter back to your HF account
4. Tests inference with personality-tagged prompts

**Requirements:** Free Colab with T4 GPU + HuggingFace account

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers>=4.44.0 datasets peft>=0.12.0 trl>=0.9.0 \
    bitsandbytes>=0.43.0 accelerate>=0.33.0 huggingface_hub torch

In [ ]:
# Cell 2: Login to HuggingFace
# Option A: Use Colab secrets (recommended)
# Go to the key icon in left sidebar -> add HF_TOKEN
# Option B: Paste token directly

from huggingface_hub import login
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    hf_token = input('Enter your HuggingFace token: ')

login(token=hf_token)
print('Logged in to HuggingFace!')

In [ ]:
# Cell 3: Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU')

## Load Training Data

Two options:
- **Option A:** Load from your HuggingFace dataset (if you ran `convert_to_chat_format.py --push-to-hf`)
- **Option B:** Upload `chat_messages_only.jsonl` directly to Colab

In [ ]:
# Cell 4: Load dataset
from datasets import load_dataset, Dataset
import json

# Option A: From HuggingFace
try:
    ds = load_dataset('issdandavis/aethermoor-chat-sft', split='train')
    # Parse messages from JSON strings
    def parse_messages(example):
        example['messages'] = json.loads(example['messages'])
        return example
    ds = ds.map(parse_messages)
    print(f'Loaded {len(ds)} records from HuggingFace')
except Exception as e:
    print(f'HF load failed ({e}), trying local file...')
    # Option B: Upload chat_messages_only.jsonl to Colab first
    from google.colab import files
    print('Upload your chat_messages_only.jsonl file:')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    records = []
    with open(fname) as f:
        for line in f:
            records.append(json.loads(line))
    ds = Dataset.from_list(records)
    print(f'Loaded {len(ds)} records from upload')

print(f'\nSample conversation:')
sample = ds[0]['messages']
for msg in sample:
    role = msg['role']
    content = msg['content'][:100] + '...' if len(msg['content']) > 100 else msg['content']
    print(f'  [{role}]: {content}')

## Load Base Model with 4-bit Quantization

QLoRA = Quantized Low-Rank Adaptation:
- Load the 3B model in 4-bit (fits in ~4GB VRAM instead of ~12GB)
- Train only small adapter layers (~1% of parameters)
- Result: Full model quality at fraction of the compute cost

In [ ]:
# Cell 5: Load model + tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_ID} in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model loaded! Parameters: {model.num_parameters():,}')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Cell 6: Configure LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA config — these adapters are what get trained
# Only ~2M parameters out of 3B — fast and cheap
lora_config = LoraConfig(
    r=16,                      # Rank (higher = more capacity, more VRAM)
    lora_alpha=32,             # Scaling factor
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Train!

SFTTrainer handles the chat template formatting automatically.
With ~500 pairs on a T4, this takes about 15-30 minutes.

In [ ]:
# Cell 7: Training
from trl import SFTConfig, SFTTrainer

# Training config — tuned for free Colab T4
training_args = SFTConfig(
    output_dir='./aethermoor-npc-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch size = 8
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    max_seq_length=2048,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_8bit',        # Memory-efficient optimizer
    gradient_checkpointing=True,      # Trade compute for VRAM
    report_to='none',                 # No wandb needed
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer,
)

print('Starting training...')
print(f'Dataset: {len(ds)} conversations')
print(f'Epochs: {training_args.num_train_epochs}')
print(f'Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print()

result = trainer.train()

print(f'\nTraining complete!')
print(f'Loss: {result.training_loss:.4f}')
print(f'Runtime: {result.metrics["train_runtime"]:.0f}s')

In [ ]:
# Cell 8: Push to HuggingFace
HF_REPO = 'issdandavis/aethermoor-npc-v1'

print(f'Pushing trained adapter to {HF_REPO}...')

# Save adapter locally first
trainer.save_model('./aethermoor-npc-final')
tokenizer.save_pretrained('./aethermoor-npc-final')

# Push to HuggingFace
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f'\nModel pushed to: https://huggingface.co/{HF_REPO}')
print('This is a LoRA adapter — it needs the base model (Qwen2.5-3B-Instruct) to run.')

## Test Inference

Let's talk to the Aethermoor NPC and see if the personality manifold works.

In [ ]:
# Cell 9: Test inference
from transformers import pipeline

pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Test prompts with different personality activations
test_conversations = [
    {
        'name': 'Curiosity + Wisdom',
        'messages': [
            {'role': 'system', 'content': 'You are an inhabitant of Aethermoor, a realm governed by the Six Sacred Tongues (KO, AV, RU, CA, UM, DR). Your current personality state [curiosity:0.8|wisdom:0.6]: Respond with wonder backed by deep knowledge.'},
            {'role': 'user', 'content': 'What is the Harmonic Wall and why does it matter?'},
        ],
    },
    {
        'name': 'Empathy + Vigilance',
        'messages': [
            {'role': 'system', 'content': 'You are an inhabitant of Aethermoor. Your current personality state [empathy:0.9|vigilance:0.5]: Respond with care and protective awareness.'},
            {'role': 'user', 'content': 'Tell me about Polly the raven. Is she safe to trust?'},
        ],
    },
    {
        'name': 'Wit + Curiosity',
        'messages': [
            {'role': 'system', 'content': 'You are an inhabitant of Aethermoor. Your current personality state [wit:0.8|curiosity:0.5]: Respond with clever humor backed by genuine interest.'},
            {'role': 'user', 'content': 'I just pulled a rare gacha character. What should I name them?'},
        ],
    },
]

print('=' * 60)
print('AETHERMOOR NPC INFERENCE TEST')
print('=' * 60)

for test in test_conversations:
    print(f'\n--- {test["name"]} ---')
    print(f'User: {test["messages"][-1]["content"]}')

    outputs = pipe(
        test['messages'],
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = outputs[0]['generated_text'][-1]['content']
    print(f'NPC: {response}')
    print()

## Game Integration Snippet

Copy this into your game to call the model from the Aethermoor engine.

In [ ]:
# Cell 10: Game integration code (copy to your game)
INTEGRATION_CODE = '''
# === Aethermoor NPC Integration ===
# Add this to demo/engine.py or wherever your game dialogue lives

from huggingface_hub import InferenceClient
from src.gacha_isekai.personality_manifold import PersonalityManifold

class AethermoorNPC:
    """NPC powered by fine-tuned model with dual-manifold personality."""

    def __init__(self, hf_repo="issdandavis/aethermoor-npc-v1"):
        self.client = InferenceClient(hf_repo)
        self.personality = PersonalityManifold()

    def respond(self, player_message: str) -> str:
        # Activate personality from context
        self.personality.activate_from_context(player_message)

        # Generate system prompt with current personality state
        system_prompt = self.personality.generate_system_prompt(
            context=player_message
        )

        # Call the model
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": player_message},
        ]

        response = self.client.chat_completion(
            messages=messages,
            max_tokens=256,
            temperature=0.7,
        )
        return response.choices[0].message.content


# Usage:
# npc = AethermoorNPC()
# reply = npc.respond("Tell me about the Sacred Tongues")
# print(reply)
'''

print(INTEGRATION_CODE)
print('\nCopy the above into your game code!')
print('The personality manifold automatically adjusts based on what the player says.')

## Next Steps

1. **More data**: Play the game -> training.py generates governance-validated pairs -> re-run this notebook
2. **Better model**: When you have 2000+ pairs, try Qwen2.5-7B (needs Colab Pro A100)
3. **Local inference**: Download the adapter, run with `llama.cpp` for offline play
4. **HF Spaces**: Deploy as a Gradio app for web-accessible NPC chat
5. **Self-play**: Have the model generate game scenarios, validate through SCBE, add to training set